### This Jupyter Notebook derives and explains how to reach the solution and state space matrix for the Clohessy-Wilshire Equations

In [15]:
import sympy as sp

t, tau = sp.symbols('t tau', real = True)
n = sp.symbols('n', positive=True)
x0, y0, z0, vx0, vy0, vz0 = sp.symbols('x0 y0 z0 vx0 vy0 vz0', real=True)
ax, ay, az = sp.symbols('a_x a_y a_z', real=True) # Chaser S/C control input accelerations (constant per time step)

x = sp.Function('x')
z = sp.Function('z')

# Cross Track Solution: z(t)

From the CHW Equations: $\ddot{z} + n^2 z = a_z$

In [16]:
z_eq = sp.Eq(z(t).diff(t, 2) + n**2*z(t), az)
z_sol = sp.dsolve(z_eq, z(t), ics={z(0): z0, z(t).diff(t).subs(t, 0): vz0}).rhs

# Along Track Solution: y(t)

From the CHW Equations: $\ddot{y} + 2nx z = a_y$

Integrate once to get $\dot{y} + 2nx = a_y t + C_0$, where $C_0 = \dot{y_0} + 2nx_0$

In [17]:
C0 = vy0 + 2*n*x0
ydot = -2*n*x(t) + ay*t + C0

# Radial Solution: x(t)

Substitute the above $\dot{y}$ solution into $\ddot{x}$:

$\ddot{x} - 3n^2x - 2n\dot{y} = a_x$

In [18]:
x_eq = sp.Eq(x(t).diff(t, 2) - 3*n**2*x(t) - 2*n*ydot, ax)
x_sol = sp.dsolve(x_eq, x(t), ics={x(0): x0, x(t).diff(t).subs(t, 0): vx0}).rhs
x_sol = sp.simplify(x_sol)

# Substitue x(t) back into y(t) to get final form

In [19]:
ydot_sol = ydot.subs(x(t), x_sol)
y_sol = y0 + sp.integrate(ydot_sol.subs(t, tau), (tau, 0, t))
y_sol = sp.simplify(y_sol)

# Final Solution to HCW

In [20]:
print(f"x(t) solution")
print(x_sol)
x_sol

x(t) solution
(a_x + 4*n**2*x0 + 2*n*(a_y*t + vy0) + (-2*a_y + n*vx0)*sin(n*t) - (a_x + 3*n**2*x0 + 2*n*vy0)*cos(n*t))/n**2


(a_x + 4*n**2*x0 + 2*n*(a_y*t + vy0) + (-2*a_y + n*vx0)*sin(n*t) - (a_x + 3*n**2*x0 + 2*n*vy0)*cos(n*t))/n**2

In [21]:
print(f"y(t) solution")
print(y_sol)
y_sol

y(t) solution
-2*a_x*t/n + 2*a_x*sin(n*t)/n**2 - 3*a_y*t**2/2 - 4*a_y*cos(n*t)/n**2 + 4*a_y/n**2 - 6*n*t*x0 - 3*t*vy0 + 6*x0*sin(n*t) + y0 + 2*vx0*cos(n*t)/n - 2*vx0/n + 4*vy0*sin(n*t)/n


-2*a_x*t/n + 2*a_x*sin(n*t)/n**2 - 3*a_y*t**2/2 - 4*a_y*cos(n*t)/n**2 + 4*a_y/n**2 - 6*n*t*x0 - 3*t*vy0 + 6*x0*sin(n*t) + y0 + 2*vx0*cos(n*t)/n - 2*vx0/n + 4*vy0*sin(n*t)/n

In [22]:
print(f"z(t) solution")
print(z_sol)
z_sol

z(t) solution
a_z/n**2 + vz0*sin(n*t)/n + (-a_z + n**2*z0)*cos(n*t)/n**2


a_z/n**2 + vz0*sin(n*t)/n + (-a_z + n**2*z0)*cos(n*t)/n**2

# Velocities for Propogation

In [23]:
print(f"vx solution")
print(x_sol.diff(t))
print(f"vy solution")
print(y_sol.diff(t))
print(f"vz solution")
print(z_sol.diff(t))

vx solution
(2*a_y*n + n*(-2*a_y + n*vx0)*cos(n*t) + n*(a_x + 3*n**2*x0 + 2*n*vy0)*sin(n*t))/n**2
vy solution
2*a_x*cos(n*t)/n - 2*a_x/n - 3*a_y*t + 4*a_y*sin(n*t)/n + 6*n*x0*cos(n*t) - 6*n*x0 - 2*vx0*sin(n*t) + 4*vy0*cos(n*t) - 3*vy0
vz solution
vz0*cos(n*t) - (-a_z + n**2*z0)*sin(n*t)/n


# Discrete Time Step: State Space Equation

In [24]:
dt = sp.symbols('dt', positive=True)
state = sp.Matrix([x_sol, y_sol, z_sol])
state = state.col_join(state.diff(t))
state_dt = state.subs(t, dt)
X0 = sp.Matrix([x0, y0, z0, vx0, vy0, vz0]); U = sp.Matrix([ax, ay, az])
A = state_dt.jacobian(X0).applyfunc(sp.simplify)
B = state_dt.jacobian(U).applyfunc(sp.simplify)

In [25]:
print(f"A Matrix (Ax + Bu)")
print(A)
A

A Matrix (Ax + Bu)
Matrix([[4 - 3*cos(dt*n), 0, 0, sin(dt*n)/n, 2*(1 - cos(dt*n))/n, 0], [-6*dt*n + 6*sin(dt*n), 1, 0, 2*(cos(dt*n) - 1)/n, -3*dt + 4*sin(dt*n)/n, 0], [0, 0, cos(dt*n), 0, 0, sin(dt*n)/n], [3*n*sin(dt*n), 0, 0, cos(dt*n), 2*sin(dt*n), 0], [6*n*(cos(dt*n) - 1), 0, 0, -2*sin(dt*n), 4*cos(dt*n) - 3, 0], [0, 0, -n*sin(dt*n), 0, 0, cos(dt*n)]])


Matrix([
[      4 - 3*cos(dt*n), 0,            0,         sin(dt*n)/n,   2*(1 - cos(dt*n))/n,           0],
[-6*dt*n + 6*sin(dt*n), 1,            0, 2*(cos(dt*n) - 1)/n, -3*dt + 4*sin(dt*n)/n,           0],
[                    0, 0,    cos(dt*n),                   0,                     0, sin(dt*n)/n],
[        3*n*sin(dt*n), 0,            0,           cos(dt*n),           2*sin(dt*n),           0],
[  6*n*(cos(dt*n) - 1), 0,            0,        -2*sin(dt*n),       4*cos(dt*n) - 3,           0],
[                    0, 0, -n*sin(dt*n),                   0,                     0,   cos(dt*n)]])

In [26]:
print(f"B Matrix (Ax + Bu)")
print(B)
B

B Matrix (Ax + Bu)
Matrix([[(1 - cos(dt*n))/n**2, 2*(dt*n - sin(dt*n))/n**2, 0], [2*(-dt*n + sin(dt*n))/n**2, (-3*dt**2*n**2 - 8*cos(dt*n) + 8)/(2*n**2), 0], [0, 0, (1 - cos(dt*n))/n**2], [sin(dt*n)/n, 2*(1 - cos(dt*n))/n, 0], [2*(cos(dt*n) - 1)/n, -3*dt + 4*sin(dt*n)/n, 0], [0, 0, sin(dt*n)/n]])


Matrix([
[      (1 - cos(dt*n))/n**2,                  2*(dt*n - sin(dt*n))/n**2,                    0],
[2*(-dt*n + sin(dt*n))/n**2, (-3*dt**2*n**2 - 8*cos(dt*n) + 8)/(2*n**2),                    0],
[                         0,                                          0, (1 - cos(dt*n))/n**2],
[               sin(dt*n)/n,                        2*(1 - cos(dt*n))/n,                    0],
[       2*(cos(dt*n) - 1)/n,                      -3*dt + 4*sin(dt*n)/n,                    0],
[                         0,                                          0,          sin(dt*n)/n]])